In [1]:
from grequests import download

download('Chapter03' , 'data01.txt')

Downloaded data01.txt to Download\data01.txt


## DATA

In [4]:
filepath = 'data01.txt'
try:
    with open(filepath , 'r') as file:
        text = file.read()
    text 
except Exception as e:
    print("Exception :- " , str(e))
print(text)

The CTO was explaing that a business-ready generative AI system (GenAISys) offers functionality similar to ChatGPT-like platforms. It combines generative AI models, RAG, memory retention, and a wide range of ML and non-AI functions managed by an AI controller. The controller orchestrates tasks dynamically rather than following the same set of instructions for each task.
GenAISys relies on a generative AI model such as GPT-4o or any advanced LLM. The CTO said that we saw that getting access to the API is insufficient. Contextual awareness and memory retention are critical components of a GenAISys. Although they are seamlessly available in ChatGPT-like platforms, we have to build them into our systems.
We defined memoryless, short-term, long-term memory, and cross-topic memory. For the hybrid travel marketing campaign, we will distinguish semantic memory(facts) from episodic memory(personal events in time, for example). The CTO said that the we will need to use episodic memories of past 

## Chunking the Dataset

In [6]:
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
import os 
openai_api = os.getenv('open_ai_api')
client = OpenAI(api_key=openai_api)

# Chunk text using ChatGPT
def chunk_text_with_gpt_40(text):
    messages = [
        {'role' : 'system' , 'content' : 'You are an assistant skilled at splitting long texts into meaningful, semantically coherent chunks of 50-100 words each.'},
        {'role' : 'user' , 'content' : f'Split the following text into meaningful chunks:\n\n{text}'}
    ]

    response = client.chat.completions.create(
        messages=messages,
        model = 'gpt-4o',
        temperature=0.2,
        max_completion_tokens = 1024
    )

    chunked_text = response.choices[0].message.content
    chunks = chunked_text.split("\n\n")  # Assume GPT-4o separates chunks with double newlines

    return chunks

# Chunk Text
chunks = chunk_text_with_gpt_40(text)

print("Chunks:")
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}:")
    print(chunk)

Chunks:

Chunk 1:
The CTO was explaining that a business-ready generative AI system (GenAISys) offers functionality similar to ChatGPT-like platforms. It combines generative AI models, RAG, memory retention, and a wide range of ML and non-AI functions managed by an AI controller. The controller orchestrates tasks dynamically rather than following the same set of instructions for each task.

Chunk 2:
GenAISys relies on a generative AI model such as GPT-4o or any advanced LLM. The CTO said that we saw that getting access to the API is insufficient. Contextual awareness and memory retention are critical components of a GenAISys. Although they are seamlessly available in ChatGPT-like platforms, we have to build them into our systems.

Chunk 3:
We defined memoryless, short-term, long-term memory, and cross-topic memory. For the hybrid travel marketing campaign, we will distinguish semantic memory (facts) from episodic memory (personal events in time, for example). The CTO said that we will 

In [17]:
# print length and content of the first 10 chunks 
for i in range(3):
    print(len(chunks[i]))
    print(chunks[i])

print("*"*113)
print(f"\nTotal number of chunks: {len(chunks)}")

374
The CTO was explaining that a business-ready generative AI system (GenAISys) offers functionality similar to ChatGPT-like platforms. It combines generative AI models, RAG, memory retention, and a wide range of ML and non-AI functions managed by an AI controller. The controller orchestrates tasks dynamically rather than following the same set of instructions for each task.
336
GenAISys relies on a generative AI model such as GPT-4o or any advanced LLM. The CTO said that we saw that getting access to the API is insufficient. Contextual awareness and memory retention are critical components of a GenAISys. Although they are seamlessly available in ChatGPT-like platforms, we have to build them into our systems.
474
We defined memoryless, short-term, long-term memory, and cross-topic memory. For the hybrid travel marketing campaign, we will distinguish semantic memory (facts) from episodic memory (personal events in time, for example). The CTO said that we will need to use episodic memor

## Embeddings the chunks

In [19]:
def get_embedding(texts, model="text-embedding-3-small"):
    texts = [text.replace("\n", " ") for text in texts]  # Clean input texts
    response = client.embeddings.create(input=texts, model=model)  # API call for batch
    embeddings = [res.embedding for res in response.data]  # Extract embeddings
    return embeddings

In [ ]:
import time 
import os 

def embed_chunks(chunks, embedding_model="text-embedding-3-small", batch_size=1000, pause_time=3):
    start_time = time.time()
    embeddings = []
    counter = 1

    for i in range(0 , len(chunks) , batch_size):

        # Select batch of chunk
        chunk_batch = chunks[i : i+batch_size]

        # create embeddings for current batch
        current_embeddings = get_embedding(chunk_batch , embedding_model)

        # Append to the main embeddings
        embeddings.extend(current_embeddings)

        # Print batch progress and pause
        print(f"Batch {counter} embedded.")
        counter += 1
        time.sleep(pause_time)
    
    response_time = time.time() - start_time
    print(f"Total Response Time: {response_time:.2f} seconds")

    return embeddings

embeddings = embed_chunks(chunks)

Batch 1 embedded.
Total Response Time: 4.84 seconds


In [21]:
print("First Embeddings :- " , embeddings[0])

First Embeddings :-  [-0.01169586181640625, 0.00701141357421875, 0.041259765625, -0.00765228271484375, 0.005420684814453125, -0.052398681640625, -0.0065765380859375, 0.02764892578125, 0.0206146240234375, -0.0277557373046875, 0.0014247894287109375, -0.05767822265625, -0.01419830322265625, -0.028656005859375, -0.04034423828125, -0.0430908203125, -0.00458526611328125, -0.04803466796875, 0.01470947265625, -0.01049041748046875, 0.0305633544921875, 0.036346435546875, -0.0158538818359375, 0.033538818359375, 0.0048675537109375, -0.019012451171875, 0.0277099609375, 0.04974365234375, -0.026275634765625, 0.0001266002655029297, 0.045013427734375, -0.017578125, -0.01233673095703125, -0.017303466796875, 0.018218994140625, 0.0158233642578125, 0.034149169921875, 0.018310546875, 0.0177459716796875, 0.0391845703125, -0.00846099853515625, -0.017181396484375, 0.01468658447265625, 0.0188446044921875, 0.0050811767578125, -0.004123687744140625, 0.0147552490234375, -0.026031494140625, -0.0219268798828125, 0.0

In [22]:
num_chunks = len(chunks)
print(f"Number of chunks: {num_chunks}")
print(f"Number of embeddings: {len(embeddings)}")

Number of chunks: 8
Number of embeddings: 8


## Pinecone Index

In [25]:
import os 
from pinecone import ServerlessSpec , Pinecone 

pinecone_api = os.getenv("pinecone_api")
pc = Pinecone(api_key=pinecone_api)

In [26]:
index_name = 'genai-v1'
namespace="data01"
cloud = os.environ.get('PINECONE_CLOUD') or 'aws'
region = os.environ.get('PINECONE_REGION') or 'us-east-1'

spec = ServerlessSpec(cloud=cloud , region=region)

In [27]:
import time 

# Check if the index avaiable or not 
if index_name not in pc.list_indexes().names():
    # If not exists
    pc.create_index(
        index_name,
        dimension=1536,
        metrics = 'cosine',
        spec=spec
    )
    print("Done now waiting to resettle.")
    time.sleep(1)

# Coneect to index 
index = pc.Index(index_name)
# Status of the index
index.describe_index_stats()


DescribeIndexStatsResponse(dimension=1536, total_vector_count=3, metric='cosine', namespaces=1)

In [28]:
import pinecone
import time
import sys

start_time = time.time()  # Start timing before the request

# Function to calculate the size of a batch
def get_batch_size(data, limit=4000000):  # limit set to 4MB to be safe
    total_size = 0
    batch_size = 0
    for item in data:
        item_size = sum([sys.getsizeof(v) for v in item.values()])
        if total_size + item_size > limit:
            break
        total_size += item_size
        batch_size += 1
    return batch_size

# Upsert function with namespace
def upsert_to_pinecone(batch, batch_size, namespace="data01"):
    """
    Upserts a batch of data to Pinecone under a specified namespace.
    """
    try:
        index.upsert(vectors=batch, namespace=namespace)
        print(f"Upserted {batch_size} vectors to namespace '{namespace}'.")
    except Exception as e:
        print(f"Error during upsert: {e}")

# Function to upsert data in batches
def batch_upsert(data):
    total = len(data)
    i = 0
    while i < total:
        batch_size = get_batch_size(data[i:])
        batch = data[i:i + batch_size]
        if batch:
            upsert_to_pinecone(batch, batch_size, namespace="data01")
            i += batch_size
            print(f"Upserted {i}/{total} items...")  # Display current progress
        else:
            break
    print("Upsert complete.")

# Generate IDs for each data item
ids = [str(i) for i in range(1, len(chunks) + 1)]

# Prepare data for upsert
data_for_upsert = [
    {"id": str(id), "values": emb, "metadata": {"text": chunk}}
    for id, (chunk, emb) in zip(ids, zip(chunks, embeddings))
]

# Upsert data in batches
batch_upsert(data_for_upsert)

response_time = time.time() - start_time  # Measure response time
print(f"Upsertion response time: {response_time:.2f} seconds")  # Print response time

Upserted 8 vectors to namespace 'data01'.
Upserted 8/8 items...
Upsert complete.
Upsertion response time: 2.04 seconds


In [30]:
print("Index stats")
print(index.describe_index_stats())

Index stats
DescribeIndexStatsResponse(dimension=1536, total_vector_count=11, metric='cosine', namespaces=2)
